In [2]:
#This shows ICA separates mixed signals.
import numpy as np
from sklearn.decomposition import FastICA
#Ceate 2 simple independent signals
S = np.array([[1,1],[2,-1],[3,1],[4,-1]])
#Mix them
A = np.array([[1,2],[3,4]])
X = S@A.T #Observed mix data
#Apply ICA
ica = FastICA(n_components=2,random_state=0)
U = ica.fit_transform(X)
print("Original Source: ",S)
print("Record Source(approx): ",U)

Original Source:  [[ 1  1]
 [ 2 -1]
 [ 3  1]
 [ 4 -1]]
Record Source(approx):  [[-1.  1.]
 [-1. -1.]
 [ 1.  1.]
 [ 1. -1.]]


In [6]:
#IMPLEMENTATION OF AUTOENCODER(ex:COMPRESSION OF ZIP FILE)
import numpy as np
from sklearn.neural_network import MLPRegressor
#Create simple data(4-features)
X = np.array([[1,2,3,4],
             [2,3,4,5],
             [3,4,5,5],
             [4,5,6,7]],dtype = float)
#CREATE AUTOENCODER(4->2->4)
autoencoder = MLPRegressor(hidden_layer_sizes=(2,),activation='relu',max_iter=5000,random_state=0)
#TRAIN TO REPRODUCE INPUT
autoencoder.fit(X,X)
#Reconstructor
X_reconstructed = autoencoder.predict(X)
#PRINT 2D HIDDEN REPRESENTATION 
W = autoencoder.coefs_[0]
b = autoencoder.intercepts_[0]
hidden_2d = np.maximum(0,X@W+b)
print("Original: ",X)
print("Hidden: ",np.round(hidden_2d,2))
print("Reconstructed: ",np.round(X_reconstructed,2))

Original:  [[1. 2. 3. 4.]
 [2. 3. 4. 5.]
 [3. 4. 5. 5.]
 [4. 5. 6. 7.]]
Hidden:  [[4.34 1.57]
 [5.81 2.12]
 [7.04 2.15]
 [8.73 3.23]]
Reconstructed:  [[2.   2.73 3.37 3.83]
 [2.46 3.29 4.05 4.84]
 [3.17 4.01 4.94 5.41]
 [3.36 4.39 5.4  6.87]]


In [10]:
#Implementation of active learning
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

#Create simple dataset
X,y = make_classification(n_samples = 100,n_features=2,n_redundant=0,n_informative = 2, random_state = 0)

#Start with 10 labeled samples
labeled = np.arange(10)
unlabeled = np.arange(10,100)

model = LogisticRegression(solver="liblinear")
for i in range(5):
    model.fit(X[labeled],y[labeled])
    probs = model.predict_proba(X[unlabeled])[:,1]
    #pick most uncertain sample
    query = np.argmin(np.abs(probs - 0.5))
    
    #move that sample to labeled set
    labeled = np.append(labeled,unlabeled[query])
    unlabeled = np.delete(unlabeled,query)
    print("Iteration: ",i+1,"| Labeled Size: ",len(labeled))
    

Iteration:  1 | Labeled Size:  11
Iteration:  2 | Labeled Size:  12
Iteration:  3 | Labeled Size:  13
Iteration:  4 | Labeled Size:  14
Iteration:  5 | Labeled Size:  15


In [11]:
#SIMPLE 1D INPUT DATA
X = np.array([[1],[2],[3],[4]])
y = np.array([0,0,1,1])

#Choose 2 fixed RBF centers manually
centers = np.array([[1],[4]])
sigma = 1.0

#Gaussian RBF function
def rbf(x,c):
    return np.exp(-np.linalg.norm(x-c)**2/(2*sigma**2))

#Build hidden layer output
H = np.array([[rbf(x,c) for c in centers] for x in X])

#Solve for weights using simple linear equation
w = np.linalg.pinv(H)@y #Moore-Penrose inverse

#Predict
y_pred = H@w
y_pred_class = (y_pred>0.5).astype(int)
print("Predicted: ",y_pred_class)

Predicted:  [0 0 1 1]
